# Phase 1 — Reconnaître les INSTRUMENTS (séparation NMF)

Un rythme de djembé = un **ensemble** : plusieurs djembés + dununs (dununba/sangban/kenkeni) + cloche.
Nos vidéos sont des **mix mono** : à chaque instant, plusieurs instruments sonnent ensemble.
Avant de reconnaître les 3 frappes du djembé, il faut donc d'abord **séparer les voix instrumentales**.

**Méthode (issue des thèses de séparation de percussions) : la NMF.**
On décompose le spectrogramme `S ≈ W · H` :
- `W` = quelques **briques spectrales** (chacune ≈ une voix : dunun grave, cloche métallique, djembé large bande)
- `H` = **quand** chaque brique joue dans le temps

Référence vérité-terrain : corpus **Malian Jembe (Polak / IEMPDC)**, enregistré en multipiste (1 micro par tambour).
Nous, on n'a que le mix → la NMF est notre façon de « démixer ».

**Plan :** 1) charger un extrait → 2) NMF → 3) **écouter** chaque brique → 4) **étiqueter** par instrument → 5) timeline des instruments → 6) carte des événements par instrument.

> Limite assumée : on séparera bien les *familles* (dunun / cloche / djembé). Distinguer les 3 dununs entre eux (juste une question de hauteur) sera plus dur.

## 0. Imports & configuration

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import librosa, librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

warnings.filterwarnings('ignore')

SR   = 22050
N_FFT, HOP = 2048, 512
AUDIO_DIR  = 'audio'
OUT_DIR    = 'output'
os.makedirs(OUT_DIR, exist_ok=True)
print('OK — librosa', librosa.__version__)

## 1. Charger un extrait d'un enregistrement

On travaille sur un extrait (l'intro contient souvent des réglages). Règle `FICHIER`, `DEBUT`, `DUREE`.

In [ ]:
FICHIER = 'Woloso'   # <- 'Djansa', 'Gede', ...
DEBUT   = 30         # s
DUREE   = 45         # s

y, _ = librosa.load(f'{AUDIO_DIR}/{FICHIER}.wav', sr=SR, mono=True,
                    offset=DEBUT, duration=DUREE)
D = librosa.stft(y, n_fft=N_FFT, hop_length=HOP)   # complexe (on garde la phase)
S = np.abs(D)
freqs = librosa.fft_frequencies(sr=SR, n_fft=N_FFT)
print(f'{FICHIER} : {DUREE}s | spectrogramme {S.shape} (freq x temps)')
display(Audio(y, rate=SR))

## 2. Décomposition NMF

`K` = nombre de briques. Trop peu → instruments mélangés ; trop → une voix éclatée en morceaux. On commence à 8 et on ajustera après écoute. Les briques sont triées du plus **grave** au plus **aigu**.

In [ ]:
K = 8   # <- nombre de voix à extraire
W, H = librosa.decompose.decompose(S, n_components=K, sort=False, random_state=0)

# trie grave -> aigu (centroïde spectral du template)
centroids = (W * freqs[:, None]).sum(0) / (W.sum(0) + 1e-9)
order = np.argsort(centroids)
W, H, centroids = W[:, order], H[order], centroids[order]

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for k in range(K):
    ax[0].semilogx(freqs[1:], W[1:, k]/(W[1:, k].max()+1e-9) + k, label=f'c{k} ({centroids[k]:.0f}Hz)')
ax[0].set(title='Templates spectraux (1 brique = 1 voix)', xlabel='Fréquence (Hz, log)',
          ylabel='composante (décalée)'); ax[0].legend(fontsize=8)
t = librosa.frames_to_time(np.arange(H.shape[1]), sr=SR, hop_length=HOP)
ax[1].imshow(H/(H.max(1, keepdims=True)+1e-9), aspect='auto', origin='lower',
             extent=[t[0], t[-1], -0.5, K-0.5], cmap='magma')
ax[1].set(title='Activations (quand chaque voix joue)', xlabel='temps (s)', ylabel='composante')
plt.tight_layout(); plt.show()

## 3. ÉCOUTER chaque composante (l'étape clé)

On resynthétise chaque brique avec un masque doux (façon Wiener) en réutilisant la phase d'origine.
**Écoute chaque extrait** et note dans ta tête : dunun ? cloche ? djembé ? (et quel type de frappe).

In [ ]:
WH = W @ H + 1e-9
comp_audio = {}
for k in range(K):
    mask = np.outer(W[:, k], H[k]) / WH
    comp_audio[k] = librosa.istft(mask * D, hop_length=HOP, length=len(y))
    print(f'--- composante c{k}  (centroïde {centroids[k]:.0f} Hz) ---')
    display(Audio(comp_audio[k], rate=SR))

## 4. Étiqueter les composantes par instrument

**À toi de jouer** : après écoute, remplis le dictionnaire ci-dessous. Plusieurs composantes peuvent recevoir le même label (ex. 2 briques = djembé).
Vocabulaire suggéré : `dunun`, `cloche`, `djembe`, `mix` (si inséparable).

In [ ]:
# <-- ÉDITE ces labels d'après ce que tu entends (c'est le coeur de la Phase 1)
LABELS = {
    0: 'dunun',
    1: 'dunun',
    2: 'djembe',
    3: 'djembe',
    4: 'djembe',
    5: 'djembe',
    6: 'cloche',
    7: 'cloche',
}
INSTR_COLORS = {'dunun':'#2b6cff', 'djembe':'#d63bd6', 'cloche':'#2ec45f', 'mix':'#888888'}
print('Instruments retenus :', sorted(set(LABELS.values())))

## 5. Timeline des instruments

On somme les activations des composantes d'un même instrument → on voit **qui joue quand**.

In [ ]:
instruments = sorted(set(LABELS.values()))
env = {ins: H[[k for k in range(K) if LABELS[k]==ins]].sum(0) for ins in instruments}

plt.figure(figsize=(14, 4))
for ins in instruments:
    e = env[ins] / (env[ins].max() + 1e-9)
    plt.plot(t, e, label=ins, color=INSTR_COLORS.get(ins, None), alpha=.8)
plt.legend(); plt.xlabel('temps (s)'); plt.ylabel('activité (norm.)')
plt.title(f'{FICHIER} — activité par instrument'); plt.xlim(0, 12); plt.show()

## 6. Onsets par instrument → carte des événements

Pour chaque instrument, on détecte les pics d'activité (= ses frappes), puis on dessine la **carte des événements** colorée par instrument. C'est le début de la « map qui reconnaît l'instrument ».

In [ ]:
rows = []
for ins in instruments:
    on = librosa.onset.onset_detect(onset_envelope=env[ins], sr=SR, hop_length=HOP,
                                    units='time', backtrack=False)
    for ot in on:
        f = int(ot*SR/HOP)
        cen = (S[:, f]*freqs).sum()/(S[:, f].sum()+1e-9) if f < S.shape[1] else 0
        rows.append({'instrument': ins, 'time': ot, 'centroid': cen})
ev = pd.DataFrame(rows)
print(ev['instrument'].value_counts())

plt.figure(figsize=(14, 4))
for ins in instruments:
    m = ev['instrument']==ins
    plt.scatter(ev.loc[m,'time'], ev.loc[m,'centroid'], s=18, alpha=.7,
                color=INSTR_COLORS.get(ins, None), label=ins)
plt.legend(); plt.xlabel('temps (s)'); plt.ylabel('centroïde spectral (Hz)')
plt.title(f'{FICHIER} — événements par instrument'); plt.show()

ev.to_csv(f'{OUT_DIR}/events_{FICHIER}.csv', index=False)
print('Sauvegardé ->', f'{OUT_DIR}/events_{FICHIER}.csv')

## Suite

- Ajuster `K` et les `LABELS` jusqu'à une séparation propre (vérifiée à l'oreille en cellule 3).
- Généraliser : appliquer la même décomposition aux 7 rythmes et construire une **carte UMAP des événements colorée par instrument**.
- **Phase 2** : isoler la voix `djembe` et y refaire l'étude fine bass/tone/slap (notebook `djembe.ipynb`).
- **Phase 3** : analyser le motif/cycle global de chaque rythme pour les différencier.